In [ ]:
# ===============================
# 各種ライブラリのインストール
# ===============================

# !pip install -r requirements.txt

In [ ]:
# ===============================
# 各種ライブラリのインポート
# ===============================

import os
import re
import sys
import shap
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from typing import List
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (RepeatedStratifiedKFold, RandomizedSearchCV, StratifiedKFold)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, 
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
# ===============================
# 特徴量エンジニアリング
# ===============================

# Name列から敬称を抽出する関数
def extract_title(name):
    TITLE_MAP = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Royal', 'Countess':'Royal','Dona':'Royal',
    'Sir':'Royal','Don':'Royal','Jonkheer':'Royal','the Countess':'Royal',
    'Capt':'Officer','Col':'Officer','Major':'Officer','Dr':'Officer','Rev':'Officer'
    }
    if not isinstance(name, str):
        return "Unknown"
    m = re.search(r",\s*([^\.]+)\.", name)
    title = m.group(1).strip() if m else "Unknown"
    return TITLE_MAP.get(title, title)


# Name列から姓（家族識別用）を抽出する関数
def extract_surname(name):
    if not isinstance(name, str):
        return "Unknown"
    return name.split(",")[0].strip()


# Cabin列の先頭文字からデッキ階層を抽出する関数
def cabin_deck(cabin):
    if not isinstance(cabin, str) or cabin.strip() == "":
        return "M"  # Missing
    return cabin.strip()[0]


# 除算（ゼロ除算防止付き）
def safe_div(a, b):
    try:
        return a / b if b and b != 0 else np.nan
    except Exception:
        return np.nan

def ticket_prefix(ticket):
    if not isinstance(ticket, str):
        return "NONE"
    # keep only letters and slashes as prefix; digits stripped
    t = re.sub(r"[./\s]", "", ticket.upper())
    m = re.match(r"[A-Z]+", t)
    return m.group(0) if m else "NONE"

def add_features(df, is_train=True):
    warnings.filterwarnings("ignore")
    df = df.copy()
    if 'Cabin' not in df.columns:
        df['Cabin'] = np.nan

    # 敬称/家族
    df['Title'] = df['Name'].apply(extract_title)
    df['Surname'] = df['Name'].apply(extract_surname)
    df['FamilySize'] = df['SibSp'].fillna(0) + df['Parch'].fillna(0) + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    # デッキ・チケット
    df['Deck'] = df['Cabin'].apply(cabin_deck)

    # --- Ticket 前処理 ---
    # プレフィックス（既存）
    df['TicketPrefix'] = df['Ticket'].apply(ticket_prefix)
    # Ticket の“出現回数”＝連続値（既存の TicketGroupSize を強化）
    ticket_counts = df['Ticket'].astype(str).value_counts()
    df['TicketGroupSize'] = df['Ticket'].astype(str).map(ticket_counts).fillna(1).astype(int)

    # Ticket の“文字数”（記号・空白除去）
    ticket_clean = df['Ticket'].astype(str).str.upper().str.replace(r"[./\s]", "", regex=True)
    df['TicketLen'] = ticket_clean.str.len().astype(int)

    # ===== ここが今回の追加：0/1/2 のビニング =====
    # 1) 出現回数のビニング（例: 1 → 0, 2〜4 → 2, 5以上 → 1）
    #    ※「2〜4は同行グループらしい」等のパターンをラベル化
    def _bin_group(n):
        if 2 <= n <= 4: return 2
        elif n >= 5:    return 1
        else:           return 0
    df['TicketGroupBin'] = df['TicketGroupSize'].apply(_bin_group).astype('category')

    # 2) 文字数のビニング（短/中/長を 0/1/2）
    #    分布が偏る場合は境界を好みに調整（例: 5, 7）
    def _bin_len(L):
        if L <= 5:  return 0
        if L <= 7:  return 1
        return 2
    df['TicketLenBin'] = df['TicketLen'].apply(_bin_len).astype('category')

    # 運賃系
    df['Fare'] = df['Fare'].replace(0, np.nan)
    df['FarePerPerson'] = df.apply(lambda r: safe_div(r.get('Fare', np.nan), r.get('FamilySize', np.nan)), axis=1)

    # 年齢系
    df['IsChild'] = (df['Age'].fillna(-1) < 16).astype(int)
    df['IsSenior'] = (df['Age'].fillna(999) >= 60).astype(int)

    # 交互作用
    df['Pclass*Sex'] = df['Pclass'].astype(str) + "_" + df['Sex'].astype(str)

    # Embarked欠損フラグ/キャビン数
    df['Embarked_isNA'] = df['Embarked'].isna().astype(int)
    df['CabinCount'] = df['Cabin'].fillna("").apply(lambda s: 0 if s=="" else len(s.split()))

    # Rare Title（既存）
    title_counts = df['Title'].value_counts()
    rare_titles = title_counts[title_counts < 10].index
    df['TitleBucket'] = df['Title'].where(~df['Title'].isin(rare_titles), 'Rare')

    # Deck のレア圧縮（任意・安定化）
    deck_counts = df['Deck'].value_counts()
    df['Deck'] = df['Deck'].fillna('M')
    rare_decks = deck_counts[deck_counts < 10].index
    df['Deck'] = df['Deck'].where(~df['Deck'].isin(rare_decks), 'Rare')

    # log1p
    for col in ['Fare', 'FarePerPerson']:
        df[f'Log1p_{col}'] = np.log1p(df[col].astype(float))

    return df


In [ ]:
# ===============================
# Target Encoding
# ===============================

class GroupAwareTargetEncoder(BaseEstimator, TransformerMixin):
    
    def __init__(self, cols: List[str], splitter=None, smoothing: float=10.0):
        self.cols = cols
        self.splitter = splitter
        self.smoothing = smoothing
        self.global_mean_ = None
        self.maps_ = {}

    
    def fit(self, X, y, groups=None):
        X = pd.DataFrame(X).copy()
        y = pd.Series(y).astype(float)
        self.global_mean_ = y.mean()
        self.maps_ = {c: {} for c in self.cols}

        # splitter が無い場合は通常KFold相当でフォールバック
        if self.splitter is None:
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            split_iter = skf.split(X, y.round().astype(int))
        else:
            # Group 対応の split（StratifiedGroupKFold or GroupKFold）
            # StratifiedGroupKFold の split は (X, y, groups) を受ける
            try:
                split_iter = self.splitter.split(X, y.round().astype(int), groups)
            except TypeError:
                split_iter = self.splitter.split(X, y, groups)

        # 各foldのtrain側統計を累積→平滑化
        sums = {c: {} for c in self.cols}
        counts = {c: {} for c in self.cols}
        for tr_idx, val_idx in split_iter:
            X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
            for c in self.cols:
                stats = y_tr.groupby(X_tr[c]).agg(['sum', 'count'])
                for k, row in stats.iterrows():
                    sums[c][k] = sums[c].get(k, 0.0) + row['sum']
                    counts[c][k] = counts[c].get(k, 0.0) + row['count']

        for c in self.cols:
            mapping = {}
            for k in counts[c].keys():
                cnt = counts[c][k]
                mean = sums[c][k] / max(cnt, 1.0)
                alpha = cnt / (cnt + self.smoothing)
                mapping[k] = alpha * mean + (1 - alpha) * self.global_mean_
            self.maps_[c] = mapping
        return self

    
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in self.cols:
            X[f"TE_{c}"] = X[c].map(self.maps_.get(c, {})).fillna(self.global_mean_)
        return X


In [ ]:
# ===============================
# 欠損値の補完（条件付き中央値）
# ===============================

def oof_conditional_median_impute(train_df, test_df, y, groups, splitter):
    """
    OOFで条件付き中央値を使って Age と Fare を埋める（リーク無し・安全版）。
    - valid側の埋めは fold 内の train 統計のみで計算（＝リーク防止）
    - test側は train 全体統計で埋める（未知データなのでOK）
    追加列:
      Age_isNA, Fare_isNA, Age_imp, Fare_imp
    """
    import numpy as np
    import pandas as pd

    tr = train_df.copy()
    te = test_df.copy()

    # ====== 前処理（キーの欠損/型ズレ対策）======
    # groupbyキーで欠損があると一致しないため、fold内で都度モード埋めするが、
    # ここでも型を合わせておく（Pclassはint、Sex/Embarked/Titleはstr）
    for df in (tr, te):
        if 'Pclass' in df.columns:
            df['Pclass'] = df['Pclass'].astype('Int64')  # 欠損許容int
        if 'Sex' in df.columns:
            df['Sex'] = df['Sex'].astype(str)
        if 'Embarked' in df.columns:
            # Embarkedの欠損は後でfold内モード埋めするが、型は先に揃える
            df['Embarked'] = df['Embarked'].astype(object).astype(str)
        if 'Title' in df.columns:
            df['Title'] = df['Title'].astype(str)

    # 欠損フラグ（モデル用に残す）
    tr['Age_isNA']  = tr['Age'].isna().astype(int)
    tr['Fare_isNA'] = tr['Fare'].isna().astype(int)
    te['Age_isNA']  = te['Age'].isna().astype(int)
    te['Fare_isNA'] = te['Fare'].isna().astype(int)

    age_oof  = tr['Age'].copy()
    fare_oof = tr['Fare'].copy()

    # ====== group-aware split ======
    try:
        split_iter = splitter.split(tr, y, groups)
    except TypeError:
        split_iter = splitter.split(tr, y)

    for tr_idx, va_idx in split_iter:
        tr_fold = tr.iloc[tr_idx].copy()
        va_fold = tr.iloc[va_idx].copy()

        # --- fold内でキーの欠損をモードで一時埋め（集計のため）---
        # （va側は reindex でマッチさせるので、ここでの埋めは不要）
        if tr_fold['Embarked'].isna().any():
            embarked_mode = tr_fold['Embarked'].mode(dropna=True)
            if not embarked_mode.empty:
                tr_fold['Embarked'] = tr_fold['Embarked'].fillna(embarked_mode.iloc[0])
        if tr_fold['Title'].isna().any():
            title_mode = tr_fold['Title'].mode(dropna=True)
            if not title_mode.empty:
                tr_fold['Title'] = tr_fold['Title'].fillna(title_mode.iloc[0])

        # ====== Fare: 条件 (Pclass, Embarked) の中央値で埋め ======
        fare_med = tr_fold.groupby(['Pclass', 'Embarked'])['Fare'].median()

        fa = va_fold.set_index(['Pclass', 'Embarked'])['Fare'].copy()
        # ★ reindex を使って必ず Series を受け取る（Indexを渡さない）
        fa = fa.fillna(fare_med.reindex(fa.index))
        # まだ欠損があれば fold内の全体中央値でフォールバック
        fa = fa.fillna(tr_fold['Fare'].median())
        fare_oof.iloc[va_idx] = fa.values

        # ====== Age: 条件 (Title, Pclass, Sex) の中央値で埋め ======
        age_med = tr_fold.groupby(['Title', 'Pclass', 'Sex'])['Age'].median()

        ag = va_fold.set_index(['Title', 'Pclass', 'Sex'])['Age'].copy()
        ag = ag.fillna(age_med.reindex(ag.index))
        ag = ag.fillna(tr_fold['Age'].median())
        age_oof.iloc[va_idx] = ag.values

    # ====== test側は train全体の統計で埋め ======
    tr_all = tr.copy()
    if tr_all['Embarked'].isna().any():
        embarked_mode_all = tr_all['Embarked'].mode(dropna=True)
        if not embarked_mode_all.empty:
            tr_all['Embarked'] = tr_all['Embarked'].fillna(embarked_mode_all.iloc[0])
    if tr_all['Title'].isna().any():
        title_mode_all = tr_all['Title'].mode(dropna=True)
        if not title_mode_all.empty:
            tr_all['Title'] = tr_all['Title'].fillna(title_mode_all.iloc[0])

    fare_med_all = tr_all.groupby(['Pclass', 'Embarked'])['Fare'].median()
    te_fa = te.set_index(['Pclass', 'Embarked'])['Fare'].copy()
    te_fa = te_fa.fillna(fare_med_all.reindex(te_fa.index))
    te_fa = te_fa.fillna(tr_all['Fare'].median())

    age_med_all = tr_all.groupby(['Title', 'Pclass', 'Sex'])['Age'].median()
    te_ag = te.set_index(['Title', 'Pclass', 'Sex'])['Age'].copy()
    te_ag = te_ag.fillna(age_med_all.reindex(te_ag.index))
    te_ag = te_ag.fillna(tr_all['Age'].median())

    # ====== 出力列として反映 ======
    tr['Fare_imp'] = fare_oof.values
    tr['Age_imp']  = age_oof.values
    te['Fare_imp'] = te_fa.values
    te['Age_imp']  = te_ag.values

    return tr, te


In [ ]:
# ===============================
# 家族が生存しているか否かを集計
# ===============================


def oof_family_survival_te(train_df, test_df, y, groups, splitter,
                           key_cols=('Surname',), m=3.0, backoff_cols=('TicketPrefix',)):
    """
    Surname（などのキー）で OOF target encoding:
      FamSR_oof: train向けOOF生存率（m平滑化）
      FamSR_te : test向け生存率（学習全体でfit, m平滑化）
    さらにバックオフ（TicketPrefixなど）も同様に作って併用可能。
    """
    import numpy as np
    import pandas as pd
    tr = train_df.copy()
    te = test_df.copy()
    y = np.asarray(y).astype(int)
    base_rate = y.mean()

    def _compute_rate(df_key, y_vec):
        # df_key: Series（キー値）, y_vec: 同じ長さの0/1
        s = pd.DataFrame({'k': df_key, 'y': y_vec})
        agg = s.groupby('k')['y'].agg(['sum', 'count'])
        rate = (agg['sum'] + m * base_rate) / (agg['count'] + m)
        return rate

    # 主キー
    ktrain = train_df[list(key_cols)].astype(str).agg('|'.join, axis=1)
    ktest  = test_df[list(key_cols)].astype(str).agg('|'.join, axis=1)

    # OOF
    fam_oof = np.zeros(len(tr), dtype=float)
    try:
        split_iter = splitter.split(tr, y, groups)
    except TypeError:
        split_iter = splitter.split(tr, y)
    for tr_idx, va_idx in split_iter:
        rate = _compute_rate(ktrain.iloc[tr_idx], y[tr_idx])
        fam_oof[va_idx] = ktrain.iloc[va_idx].map(rate).fillna(base_rate).values

    # test（全体fit）
    rate_all = _compute_rate(ktrain, y)
    fam_te = ktest.map(rate_all).fillna(base_rate).values

    tr['FamSR'] = fam_oof
    te['FamSR']  = fam_te

    # ---- バックオフ（任意：TicketPrefix などで同様に）----
    for bcol in backoff_cols:
        ktrain_b = train_df[bcol].astype(str)
        ktest_b  = test_df[bcol].astype(str)
        famb_oof = np.zeros(len(tr), dtype=float)
        try:
            split_iter = splitter.split(tr, y, groups)
        except TypeError:
            split_iter = splitter.split(tr, y)
        for tr_idx, va_idx in split_iter:
            rate = _compute_rate(ktrain_b.iloc[tr_idx], y[tr_idx])
            famb_oof[va_idx] = ktrain_b.iloc[va_idx].map(rate).fillna(base_rate).values
        rate_all_b = _compute_rate(ktrain_b, y)
        famb_te = ktest_b.map(rate_all_b).fillna(base_rate).values

        tr['BackoffSR'] = famb_oof
        te['BackoffSR']  = famb_te

    return tr, te


In [ ]:
#  ===============================
# 前処理
# ===============================

# 警告防止用のダミー
class IdentityTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.feature_names_in_ = getattr(X, "columns", None)
        return self
    def transform(self, X):
        return X
    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.array(input_features)
        elif hasattr(self, "feature_names_in_") and self.feature_names_in_ is not None:
            return np.array(self.feature_names_in_)
        else:
            return np.array([])

def build_preprocessor(numeric_features, categorical_features, scaler=True):
    """
    数値とカテゴリ列をまとめて前処理する ColumnTransformer を構築する。
    - 数値列: 標準化 or そのまま通す
    - カテゴリ列: One-Hotエンコード（未知カテゴリ無視）
    """

    # 数値の前処理
    if scaler:
        numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
    else:
        # スケーリングをしない場合（ツリー系モデル向け）
        numeric_transformer = Pipeline(steps=[('identity', IdentityTransformer())])

    # カテゴリの前処理（疎行列出力を抑制）

    try:
        categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:  # sklearn <1.2
        categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse=False)

    # ColumnTransformerにまとめる
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop'  # その他の列は無視
    )

    return preprocessor


In [ ]:
# ===============================
# モデル群の定義
# ===============================

# 主要モデル群とパラメータ探索範囲を返す関数
def available_models(RANDOM_STATE):
    models = {}
    warnings.filterwarnings("ignore")

    # 1) RandomForest
    models['rf'] = (RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1), {
        'n_estimators': list(range(10, 31, 1)),
        'max_depth': [None] + list(range(3, 8)),
        'min_samples_split': list(range(2, 5)),
        'min_samples_leaf': list(range(1, 5)),
        'max_features': ['sqrt', 'log2', None],
    })

    # 2) XGBoost
    models['xgb'] = (XGBClassifier(
        n_estimators=120,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ), {
        'learning_rate': np.linspace(0.01, 0.2, 20),
        'max_depth': list(range(3, 8)),
        'min_child_weight': list(range(1, 8)),
        'subsample': np.linspace(0.6, 1.0, 9),
        'colsample_bytree': np.linspace(0.6, 1.0, 9),
        'reg_lambda': np.linspace(0.0, 5.0, 21),
        'reg_alpha': np.linspace(0.0, 2.0, 11),
        'gamma': np.linspace(0.0, 2.0, 11),
    })

    # HistGradientBoosting
    models['hgb'] = (HistGradientBoostingClassifier(random_state=RANDOM_STATE), {
        'learning_rate': np.linspace(0.02, 0.2, 10),
        'max_depth': list(range(3, 8))
    })

    models['knn'] = (KNeighborsClassifier(),{
        'n_neighbors': list(range(3, 51, 2)),
        'weights': ['uniform', 'distance'],
        'p': [1, 2]
    })

    models['svc'] = (SVC(probability=True, random_state=RANDOM_STATE),{
        'C': np.logspace(-2, 2, 50),
                         'gamma': np.logspace(-4, 0, 50),
                         'kernel': ['rbf']
                     })

    return models


In [ ]:
# OOF確率 p_oof と真値 y_true から、Accuracy を最大化する閾値を探索する。
def find_best_threshold(p_oof, y_true, t_min=0.2, t_max=0.8, steps=1000):
    ts = np.linspace(t_min, t_max, steps)
    # ベクトル化して一気に精度計算（N×K の比較をブロードキャスト）
    preds = (p_oof[:, None] >= ts[None, :]).astype(np.int32)
    accs = (preds == y_true[:, None]).mean(axis=0)
    best_idx = int(np.argmax(accs))
    return float(ts[best_idx]), float(accs[best_idx])


In [ ]:
# ===============================
# SHAPでどの特徴量が効いているか見る
# ===============================

def shap_generate_reports(model, X_df, y_vec=None, model_name="model", outdir="shap_out",
                          max_display=20, example_index=0):
    """
    与えられた学習済みモデルと学習データ DataFrame から、SHAP図をPNG保存する。
    - summary_plot (全体重要度/分布)
    - dependence_plot（上位特徴を1つ）
    - waterfall_plot（単一サンプル）
    """

    os.makedirs(outdir, exist_ok=True)

    # LightGBM/XGBoost/CatBoost/Tree系は TreeExplainer が高速
    explainer = None
    try:
        explainer = shap.TreeExplainer(model)
    except Exception:
        # だめなら汎用Explainerにフォールバック
        explainer = shap.Explainer(model, X_df)

    shap_values = explainer(X_df)

    # 1) summary（重要度＋分布）
    plt.figure()
    shap.summary_plot(shap_values, X_df, max_display=max_display, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{model_name}_summary.png"), dpi=200)
    plt.close()

    # 2) dependence（上位1特徴）
    # shap_values が pandas 互換の .values を持つ場合があるので吸収
    # 上位特徴推定：mean(|SHAP|)が最大の列
    vals = np.abs(shap_values.values if hasattr(shap_values, "values") else shap_values).mean(0)
    top_idx = int(np.argmax(vals))
    top_feat = X_df.columns[top_idx]

    plt.figure()
    shap.dependence_plot(top_feat, shap_values.values if hasattr(shap_values, "values") else shap_values,
                         X_df, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{model_name}_dependence_{top_feat}.png"), dpi=200)
    plt.close()

    # 3) waterfall（単一サンプル）
    # インデックスが範囲外の時は0にする
    if not (0 <= example_index < len(X_df)):
        example_index = 0
    plt.figure()
    shap.waterfall_plot(shap_values[example_index], show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{model_name}_waterfall_idx{example_index}.png"), dpi=200)
    plt.close()


In [ ]:
def main():

    # データの読み込み
    train = pd.read_csv('../input/titanic/train.csv')
    test = pd.read_csv('../input/titanic/test.csv')

    # シード値固定
    RANDOM_STATE = 0
    np.random.seed(RANDOM_STATE)

    # 特徴量生成
    train_fe = add_features(train, True)
    test_fe = add_features(test, False)
    y = train_fe['Survived'].astype(int)
    groups = train_fe['Ticket'].astype(str).values
    
    
    # Target Encoding
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    te_cols = ['Surname','Ticket','Cabin']
    te = GroupAwareTargetEncoder(cols=te_cols, splitter=splitter, smoothing=10.0)
    te.fit(train_fe[te_cols], y, groups=groups)
    train_te = te.transform(train_fe[te_cols])
    test_te = te.transform(test_fe[te_cols])
    for c in te_cols:
        train_fe[f'TE_{c}'] = train_te[f'TE_{c}']
        test_fe[f'TE_{c}'] = test_te[f'TE_{c}']

    # 欠損値の補完
    train_fe, test_fe = oof_conditional_median_impute(train_fe, test_fe, y.values, groups, splitter)
    # 家族単位で集計して新しい特徴量を作る
    train_fe, test_fe = oof_family_survival_te(train_fe, test_fe, y, groups, splitter,
                                               key_cols=('Surname',), m=3.0, backoff_cols=('TicketPrefix',))

    bins = [-np.inf, 5, 12, 19, 29, 44, 59, np.inf]
    labels = ['Infant(0-5)','Child(6-12)','Teen(13-19)','Young(20-29)',
              'Adult(30-44)','Mid(45-59)','Senior(60+)']
    
    for df in (train_fe, test_fe):
        df['AgeBin'] = pd.cut(df['Age_imp'], bins=bins, labels=labels, right=True, include_lowest=True)
        df['Age*Fare'] = df['Age_imp'] * (np.max(df['Fare_imp']) / df['Fare_imp'])

    
    bins = [-np.inf, 7.91, 14.454, 31.0, 99.0, 250.0, np.inf]
    labels = ['VeryLow', 'Low', 'Middle', 'UpperMiddle', 'High', 'Luxury']

    for df in (train_fe, test_fe):
        df['FareBin'] = pd.cut(df['Fare_imp'], bins=bins, labels=labels, include_lowest=True)
    
    numeric = ['FamSR','BackoffSR','Age*Fare']
    
    categorical = ['Sex','Title','Pclass',
                   'Pclass*Sex','FamilySize','IsAlone',
                   'TicketGroupBin','AgeBin','FareBin']
    
    # 前処理器を構築
    pre = build_preprocessor(numeric, categorical)

    # fit/transform
    X = pre.fit_transform(train_fe[numeric + categorical])
    X_test = pre.transform(test_fe[numeric + categorical])

    # モデル群を学習
    models = available_models(RANDOM_STATE)
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)

    best_models = {}
    for name, (model, space) in models.items():
        print(f"random search of {name} now...", flush=True)
        search = RandomizedSearchCV(model, space, n_iter=20, cv=rskf,
                                   scoring='accuracy', random_state=RANDOM_STATE, n_jobs=-1)
        search.fit(X, y)
        best_models[name] = search.best_estimator_
        print(f"  CV精度: {search.best_score_:.4f}", flush=True)

    # ====== SHAP レポート生成（学習後・メタ前）======
    try:
        feature_names = pre.get_feature_names_out()
    except Exception:
        # 古いsklearn用フォールバック：numはそのまま、catはOHE名を取得
        feature_names = numeric + list(pre.named_transformers_['cat'].get_feature_names_out(categorical))

    # X と X_testをDataFrame化
    X_df = pd.DataFrame(X, columns=feature_names)
    
    num_ok = []
    for c in X_df.columns:
        if X_df[c].dtype.kind in "biufc":       # bool,int,unsigned,float,complex
            num_ok.append(c)
        else:
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
            num_ok.append(c)
    
    X_df = X_df[num_ok].astype("float64") 

    shap_targets = []
    if 'xgb' in best_models:
        shap_targets.append(('xgb', best_models['xgb'], X_df)) 
    elif 'rf' in best_models:
        shap_targets.append(('rf', best_models['rf'], X_df))
    elif 'hgb' in best_models:
        shap_targets.append(('hgb', best_models['hgb'], X_df)) 

    # 実際に図を吐き出す（shap_out/ にPNG保存）
    for mname, mobj, Xuse in shap_targets:
        print(f"Generating SHAP Report: {mname}", flush=True)
        shap_generate_reports(mobj, Xuse, y_vec=y, model_name=mname, outdir="shap_out",
                              max_display=20, example_index=0)
        print(f"  -> shap_out/{mname}_summary.png", flush=True)

    # 上位モデルをスタッキング
    print("stacking now...", flush=True)
    oof_preds = []
    test_preds = []
    for name, model in best_models.items():
        oof = np.zeros(len(y))
        preds = []
        for tr, val in rskf.split(X, y):
            m = clone(model)
            m.fit(X[tr], y[tr])
            oof[val] = m.predict_proba(X[val])[:, 1]
            preds.append(m.predict_proba(X_test)[:, 1])
        oof_preds.append(oof)
        test_preds.append(np.mean(preds, axis=0))
    oof_stack = np.column_stack(oof_preds)
    test_stack = np.column_stack(test_preds)

    # CVで L2 強さ(C)を自動チューニング（= リッジロジスティック）
    cv_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    meta = LogisticRegressionCV(
        Cs=np.logspace(-3, 2, 25),      # C を広めに探索（10^-3 ~ 10^2）
        cv=cv_meta,                     # OOFスタック上で内側CV
        penalty='l1',                   # ← リッジ（L2）
        solver='liblinear',             # 小規模データで安定
        scoring='accuracy',             # Acc最大のCを選ぶ（Kaggle指標と整合）
        max_iter=5000,
        n_jobs=-1,
        refit=True)

    meta_cal = CalibratedClassifierCV(meta, cv=rskf)
    meta_cal.fit(oof_stack, y)
    test_final = meta_cal.predict_proba(test_stack)[:, 1]
    p_oof = np.zeros(len(y))
    p_test_folds = []
    
    for fold, (tr_idx, va_idx) in enumerate(cv_meta.split(oof_stack, y), 1):
        X_tr, X_va = oof_stack[tr_idx], oof_stack[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        meta.fit(X_tr, y_tr)
        p_oof[va_idx] = meta.predict_proba(X_va)[:, 1]
        p_test_folds.append(meta.predict_proba(test_stack)[:, 1])

    p_test = np.mean(p_test_folds, axis=0)
    
    # しきい値最適化
    best_t, best_acc = find_best_threshold(p_oof, y.values, t_min=0.2, t_max=0.8, steps=2001)
    print(f"best_acc={best_acc:.4f}", flush=True)


    # 提出ファイル作成
    sub = pd.DataFrame({
        'PassengerId': test['PassengerId'],
        'Survived': (test_final >= best_t).astype(int)
    })
    sub.to_csv('submission.csv', index=False)
    print("saved submission.csv", flush=True)

if __name__ == "__main__":
    main()